# BayMorph — simulated-galaxy validation

End-to-end validation of a simulated single-Sérsic galaxy using the same public pipeline used for observed data. The final section converts the BayMorph posterior into the permanent `baymorph.validation` recovery representation.

## Setup

In [ ]:
from pathlib import Path
import os

import numpy as np

from baymorph import (
    BayMorphRunner,
    load_resolved_config,
    run_simulation_from_config,
)
from baymorph.plotting.report import plot_image_preparation_report
from baymorph.validation import (
    GroundTruth,
    RecoveryResult,
    recovery_fit_from_baymorph,
)

if Path.cwd().name == "notebooks":
    os.chdir("..")

CONFIG_PATH = Path("configs/simulated_galaxy.yaml")
config = load_resolved_config(CONFIG_PATH)

print(f"Configuration: {CONFIG_PATH}")
print(f"Object: {config['object']['name']}")

## 1. Generate the simulated observation

In [ ]:
synthetic_observation = run_simulation_from_config(config)
synthetic_observation

## 2. Prepare the image

In [ ]:
runner = BayMorphRunner.from_config(config)
products = runner.prepare_image(return_result=True)

print("Mode:", products.cutout_mode)
print("Preliminary shape:", products.preliminary_cutout.shape)
print("Final shape:", products.cutout.shape)
print("Bounds:", products.cutout_bounds)
print("Plateau radius:", products.plateau.radius)
print("Sky pixels:", products.sky_region.n_pixels)
print("Background:", products.background)
print("Median noise:", float(np.nanmedian(products.noise)))

In [ ]:
preparation_report = (
    Path(config["outputs"]["root"])
    / config["object"]["name"]
    / "image_preparation_report.pdf"
)

plot_image_preparation_report(
    products,
    output_path=preparation_report,
)

print("Saved:", preparation_report)

## 3. Fit the configured model

In [ ]:
fit_products = runner.fit(return_result=True)
fit_products

## 4. Generate diagnostics

In [ ]:
runner.diagnostics()

## 5. Inspect the fit and quantify recovery

In [ ]:
result = runner.results

print("Best-fit parameters:")
for name, value in result.best_parameters.items():
    print(f"{name:>14s}: {value:.8g}")

print("\nFit statistics:")
for name, value in result.statistics.items():
    print(f"{name:>14s}: {value:.8g}")

In [ ]:
truth_cfg = config["simulation"]["truth"]
truth = GroundTruth(
    model_name="single_sersic",
    parameters={
        "flux": truth_cfg["flux"],
        "re": truth_cfg["re"],
        "n": truth_cfg["n"],
        "q": truth_cfg["q"],
        "theta": np.deg2rad(truth_cfg["theta_deg"]),
        "x0": truth_cfg["x0"],
        "y0": truth_cfg["y0"],
        "sky": config["simulation"]["noise"]["sky"],
    },
    image_shape=tuple(config["simulation"]["shape"]),
    pixel_scale=config["data"].get("pixel_scale"),
    sky_level=config["simulation"]["noise"]["sky"],
    random_seed=config["simulation"]["seed"],
    psf_metadata=config["simulation"]["psf"],
    noise_metadata=config["simulation"]["noise"],
)

normalized_fit = recovery_fit_from_baymorph(result)
recovery = RecoveryResult(
    case_id=config["object"]["name"],
    truth=dict(truth.parameters),
    estimates=dict(normalized_fit.estimates),
    intervals=(
        None
        if normalized_fit.intervals is None
        else dict(normalized_fit.intervals)
    ),
    log_evidence=normalized_fit.log_evidence,
    log_evidence_error=normalized_fit.log_evidence_error,
    likelihood_evaluations=normalized_fit.likelihood_evaluations,
    metadata=dict(normalized_fit.metadata),
)

print(f"{'parameter':>14s} {'truth':>14s} {'median':>14s} {'bias':>14s} {'covered':>10s}")
for name in recovery.parameter_names:
    parameter = recovery.parameter(name)
    print(
        f"{name:>14s} "
        f"{parameter.truth:14.6g} "
        f"{parameter.estimate:14.6g} "
        f"{parameter.bias:14.6g} "
        f"{str(parameter.covered):>10s}"
    )